# 📰 AG News — Exploratory Data Analysis
**Author:** Harshavarthan | Data Scientist @ Strand Life Sciences

This notebook explores the AG News dataset before fine-tuning DistilBERT.
We look at:
- Class distribution
- Text length statistics
- Word frequency per class
- Sample texts per category

In [ ]:
import sys
import os

# Make sure the project root is on the path regardless of where you run the notebook from
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

from data.dataset import load_agnews, get_label_map, preprocess_text

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120

## 1. Load Data

In [ ]:
texts, labels = load_agnews(split='train')
label_map = get_label_map()

df = pd.DataFrame({'text': texts, 'label': labels})
df['label_name'] = df['label'].map(label_map)
df['clean_text'] = df['text'].apply(preprocess_text)
df['text_length'] = df['clean_text'].apply(len)
df['word_count'] = df['clean_text'].apply(lambda x: len(x.split()))

print(f'Total samples: {len(df):,}')
df.head()

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count bar
counts = df['label_name'].value_counts()
axes[0].bar(counts.index, counts.values, color=sns.color_palette('Set2', 4))
axes[0].set_title('Class Distribution (Count)')
axes[0].set_ylabel('Number of Samples')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontsize=10)

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set2', 4), startangle=140)
axes[1].set_title('Class Distribution (%)')

plt.suptitle('AG News — Class Balance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/eda_class_distribution.png', bbox_inches='tight')
plt.show()
print('\n✅ Perfectly balanced dataset — no class weighting needed!')

## 3. Text Length Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Word count distribution
for label_name in df['label_name'].unique():
    subset = df[df['label_name'] == label_name]['word_count']
    axes[0].hist(subset, bins=40, alpha=0.6, label=label_name)
axes[0].set_title('Word Count Distribution by Class')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].axvline(x=128, color='red', linestyle='--', label='max_length=128 tokens')

# Box plot
df.boxplot(column='word_count', by='label_name', ax=axes[1])
axes[1].set_title('Word Count Spread per Class')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Word Count')
plt.suptitle('')

plt.tight_layout()
plt.savefig('../outputs/eda_text_length.png', bbox_inches='tight')
plt.show()

print(df.groupby('label_name')['word_count'].describe().round(1))

## 4. Top Words per Class

In [ ]:
STOPWORDS = set([
    'the','a','an','in','of','to','and','is','it','for','on',
    'at','by','as','with','that','this','was','are','be','from',
    'its','has','have','had','he','she','his','her','their','said'
])

def get_top_words(texts, n=15):
    words = []
    for t in texts:
        words.extend([w.lower() for w in re.findall(r'\b[a-zA-Z]{3,}\b', t)
                       if w.lower() not in STOPWORDS])
    return Counter(words).most_common(n)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for label_id, label_name in label_map.items():
    subset = df[df['label'] == i]['clean_text'].tolist()
    top_words = get_top_words(subset)
    words, counts = zip(*top_words)
    axes[label_id].barh(words[::-1], counts[::-1],
                 color=sns.color_palette('Set2', 4)[label_id])
    axes[label_id].set_title(f'Top Words — {label_name}', fontweight='bold')
    axes[label_id].set_xlabel('Frequency')

plt.suptitle('Most Frequent Words per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/eda_top_words.png', bbox_inches='tight')
plt.show()

## 5. Sample Texts per Class

In [ ]:
print('=' * 70)
for label_id, label_name in label_map.items():
    samples = df[df['label'] == label_id]['text'].sample(3, random_state=42).tolist()
    print(f'\n🏷️  {label_name.upper()}')
    print('-' * 50)
    for s in samples:
        print(f'  • {s[:120]}...')
print('\n' + '=' * 70)

## 6. Key Takeaways

| Insight | Impact on Modelling |
|---------|---------------------|
| Dataset is **perfectly balanced** (30k each) | No class weighting needed |
| Avg word count ~43, max ~250 | `max_length=128` covers >95% of samples |
| Distinctive vocabulary per class | Even bag-of-words should perform well |
| No missing values | No imputation needed |